
# 24 — RLHF with PPO and KL Regularization

## Goal

Connect supervised fine-tuning, preference reward modeling,
reference-policy regularization, and PPO into one RLHF pipeline.

We will reuse the PPO mechanisms from Lesson 21 and the
post-training primitives from Lessons 22 and 23.

## 01. RLHF Objective and Model Roles

### Why not maximize reward alone?

A reward model is a learned approximation of human preferences.
Its scores can be imperfect or exploitable.

Optimizing only the reward can push a policy toward outputs
that score highly without being genuinely useful.

A reference policy supplies an anchor:

$$
J(\theta)
=
\mathbb E_{x \sim \mathcal D,\; y \sim \pi_\theta(\cdot \mid x)}
[r_\phi(x,y)]
-
\beta\,\mathbb E_{x\sim\mathcal D}
[D_{\mathrm{KL}}(
\pi_\theta(\cdot \mid x)
\Vert
\pi_{\mathrm{ref}}(\cdot \mid x)
)]
$$

Here, beta controls the trade-off between reward and
deviation from the reference distribution.

```mermaid
flowchart TD
    A["Prompt x"] --> B["Trainable Actor"]
    B --> C["Sample response y"]
    C --> D["Frozen Reward Model"]
    C --> E["Frozen Reference Policy"]
    D --> F["Reward r(x,y)"]
    E --> G["Reference log-probabilities"]
    B --> H["Rollout-time log-probabilities"]
    G --> I["KL-adjusted rewards"]
    H --> I
    F --> I
    I --> J["GAE + Value targets"]
    K["Trainable Critic"] --> J
    J --> L["PPO updates"]
    H --> L
    L --> B
    L --> K
```

The reference policy and PPO's old policy are different.

The reference policy anchors the actor to its SFT initialization.
The old policy records the behavior that generated a particular
rollout, allowing PPO to compute importance ratios.

The Critic estimates expected future returns. It is not the
preference reward model.


In [1]:
import torch

# (Y=2,): frozen reference probabilities for responses A and B
reference_probs = torch.tensor([0.90, 0.10], dtype=torch.float64)

# (Y=2,): hypothetical reward scores for A and B
response_rewards = torch.tensor([0.0, 1.0], dtype=torch.float64)

beta = 0.5

# (N=3,): probability of choosing response B under three policies
candidate_b_probs = torch.tensor([0.10, 0.45, 0.90], dtype=torch.float64)

# (N=3, Y=2): complete distributions over A and B
candidate_policies = torch.stack([1.0 - candidate_b_probs, candidate_b_probs], dim=-1)

# (N=3,): expected reward under each candidate policy
expected_rewards = (candidate_policies * response_rewards).sum(dim=-1)

# (N=3,): exact forward KL to the reference policy
exact_kls = (
    candidate_policies * (candidate_policies.log() - reference_probs.log())
).sum(dim=-1)

# (N=3,): KL-regularized RLHF objectives
rlhf_objectives = expected_rewards - beta * exact_kls

for p_b, reward, kl, objective in zip(
    candidate_b_probs.tolist(),
    expected_rewards.tolist(),
    exact_kls.tolist(),
    rlhf_objectives.tolist(),
):
    print(
        f"P(B)={p_b:.2f} | "
        f"Reward={reward:.3f} | "
        f"KL={kl:.3f} | "
        f"Objective={objective:.3f}"
    )

# The intermediate policy can outperform both extremes.
assert rlhf_objectives[1] > rlhf_objectives[0]
assert rlhf_objectives[1] > rlhf_objectives[2]

print("KL-regularized reward trade-off verified.")


P(B)=0.10 | Reward=0.100 | KL=0.000 | Objective=0.100
P(B)=0.45 | Reward=0.450 | KL=0.406 | Objective=0.247
P(B)=0.90 | Reward=0.900 | KL=1.758 | Objective=0.021
KL-regularized reward trade-off verified.


In [2]:
# (Y=2,): exact KL-regularized optimal policy
optimal_policy = torch.softmax(reference_probs.log() + response_rewards / beta, dim=0)

# (N=999,): candidate probabilities for response B
probability_grid = torch.linspace(0.001, 0.999, steps=999, dtype=torch.float64)

# (N=999, Y=2): candidate policies on the simplex
policy_grid = torch.stack([1.0 - probability_grid, probability_grid], dim=-1)

# (N=999,): expected reward for each candidate policy
reward_grid = (policy_grid * response_rewards).sum(dim=-1)

# (N=999,): exact KL for each candidate policy
kl_grid = (policy_grid * (policy_grid.log() - reference_probs.log())).sum(dim=-1)

# (N=999,): regularized objectives
objective_grid = reward_grid - beta * kl_grid

# (): maximizing grid index
best_index = objective_grid.argmax()

# (): approximate best P(B) found by grid search
grid_optimal_p_b = probability_grid[best_index]

torch.testing.assert_close(grid_optimal_p_b, optimal_policy[1], atol=0.001, rtol=0)

torch.testing.assert_close(
    reference_probs.sum(), torch.tensor(1.0, dtype=torch.float64)
)

assert torch.all(kl_grid >= -1e-12)

print("Analytical optimal policy:", optimal_policy.tolist())
print("Grid-search optimal P(B):", grid_optimal_p_b.item())
print("Analytical and numerical solutions agree.")


Analytical optimal policy: [0.5491469396207161, 0.45085306037928385]
Grid-search optimal P(B): 0.451
Analytical and numerical solutions agree.


In [3]:
import plotly.graph_objects as go

tradeoff_fig = go.Figure()

tradeoff_fig.add_trace(
    go.Scatter(
        x=probability_grid.tolist(),
        y=reward_grid.tolist(),
        mode="lines",
        name="Expected reward",
    )
)

tradeoff_fig.add_trace(
    go.Scatter(
        x=probability_grid.tolist(),
        y=(beta * kl_grid).tolist(),
        mode="lines",
        name="KL penalty",
    )
)

tradeoff_fig.add_trace(
    go.Scatter(
        x=probability_grid.tolist(),
        y=objective_grid.tolist(),
        mode="lines",
        name="RLHF objective",
    )
)

tradeoff_fig.add_vline(
    x=optimal_policy[1].item(),
    line_dash="dash",
    annotation_text="Optimal policy",
)

tradeoff_fig.add_vline(
    x=reference_probs[1].item(),
    line_dash="dot",
    annotation_text="Reference policy",
)

tradeoff_fig.update_layout(
    title="Exact RLHF Objective — Reward vs KL Regularization",
    xaxis_title="Probability of response B",
    yaxis_title="Expected value",
    height=460,
    hovermode="x unified",
)

tradeoff_fig.show()


In [4]:
# (K=3,): different KL-regularization strengths
beta_values = torch.tensor([0.1, 0.5, 1.0], dtype=torch.float64)

# (K=3, Y=2): exact optimal policy for each beta
optimal_policies = torch.softmax(
    reference_probs.log().unsqueeze(0)
    + response_rewards.unsqueeze(0) / beta_values.unsqueeze(1),
    dim=-1,
)

for beta_value, policy in zip(beta_values.tolist(), optimal_policies.tolist()):
    print(f"beta={beta_value:.1f} | P(A)={policy[0]:.4f} | P(B)={policy[1]:.4f}")

# Stronger KL regularization keeps the optimum closer to the reference.
assert optimal_policies[0, 1] > optimal_policies[1, 1]
assert optimal_policies[1, 1] > optimal_policies[2, 1]

# beta=0.5 must agree with our earlier solution.
torch.testing.assert_close(optimal_policies[1], optimal_policy)

print("KL-strength sensitivity passed.")


beta=0.1 | P(A)=0.0004 | P(B)=0.9996
beta=0.5 | P(A)=0.5491 | P(B)=0.4509
beta=1.0 | P(A)=0.7680 | P(B)=0.2320
KL-strength sensitivity passed.



### RLHF Objective and Model Roles — Takeaways

RLHF can optimize expected preference reward while penalizing
deviation from a fixed reference policy.

The trainable actor, frozen reference policy, frozen reward
model, trainable critic, and rollout-time old policy have
distinct responsibilities.

The PPO old policy is not the reference policy.

PPO clipping controls the change relative to collected
rollout probabilities. Reference KL regularization anchors
the policy to an independently fixed baseline.

In a finite action space with fixed rewards, we can compute
the exact KL-regularized objective and solve for the optimum:

$$
\pi^*(y)
\propto
\pi_{\mathrm{ref}}(y)\exp(r(y)/\beta)
$$

We verified the analytical solution with numerical grid search.

Increasing beta makes the optimum stay closer to the reference
distribution in our two-response example.

Real autoregressive language models cannot enumerate all
possible completions, motivating sampled rollout estimation.

The next section will distinguish exact KL divergence from
sampled log-probability ratios.



## 02. Reference Policy and Sampled KL

### Exact KL versus a sampled log-ratio

For two distributions p and q with positive probabilities:

$$
D_{\mathrm{KL}}(p \Vert q)
=
\sum_y p(y)\log\frac{p(y)}{q(y)}
=
\mathbb E_{Y\sim p}
\left[\log\frac{p(Y)}{q(Y)}\right]
$$

Define the pointwise log-ratio:

$$
\ell(y)=\log p(y)-\log q(y)
$$

An individual log-ratio can be positive, zero, or negative.

Its expectation under p is the forward KL and is nonnegative.
Its expectation under q is the negative reverse KL:

$$
\mathbb E_{Y\sim q}[\ell(Y)]
=
-D_{\mathrm{KL}}(q\Vert p)
$$

```mermaid id="6h795s"
flowchart TD
    A["Current policy p"] --> B["Sample Y from p"]
    B --> C["log p(Y) - log q(Y)"]
    C --> D["One signed sample log-ratio"]
    D --> E["Average many on-policy samples"]
    E --> F["Estimate forward KL: D_KL(p || q)"]
    G["Reference policy q"] --> C
```

The distribution used to sample Y determines which expectation
the empirical average estimates.

During on-policy collection, the behavior policy is the
rollout-time policy, also called the PPO old policy.

The reference policy is a separate frozen baseline.


In [5]:
# (Y=2,): the analytically optimal policy from Section 01
policy_probs = optimal_policy.detach().clone()

# (Y=2,): pointwise log p(y) / q(y)
log_ratio_by_response = policy_probs.log() - reference_probs.log()

# (Y=2,): individual contributions to the exact forward KL
weighted_kl_terms = policy_probs * log_ratio_by_response

# (): exact forward KL from current policy to reference
exact_forward_kl = weighted_kl_terms.sum()

# (): exact reverse KL from reference to current policy
exact_reverse_kl = (
    reference_probs * (reference_probs.log() - policy_probs.log())
).sum()

for name, log_ratio, contribution in zip(
    ("A", "B"), log_ratio_by_response.tolist(), weighted_kl_terms.tolist()
):
    print(
        f"{name} | log-ratio={log_ratio:+.4f} | "
        f"weighted contribution={contribution:+.4f}"
    )

assert log_ratio_by_response[0] < 0
assert log_ratio_by_response[1] > 0
assert exact_forward_kl > 0
assert exact_reverse_kl > 0

print("Exact forward KL:", exact_forward_kl.item())
print("Exact reverse KL:", exact_reverse_kl.item())

A | log-ratio=-0.4940 | weighted contribution=-0.2713
B | log-ratio=+1.5060 | weighted contribution=+0.6790
Exact forward KL: 0.40767741271438884
Exact reverse KL: 0.29402870804417885


In [6]:
num_samples = 30_000

# A dedicated generator makes the experiment reproducible.
sampling_generator = torch.Generator().manual_seed(2402)

# (N=30000,): response IDs sampled from the current policy
policy_sample_ids = torch.multinomial(
    policy_probs, num_samples, replacement=True, generator=sampling_generator
)

# (N=30000,): signed log-ratios from sampled responses
policy_sample_log_ratios = log_ratio_by_response[policy_sample_ids]

# (): Monte Carlo estimate of forward KL
estimated_forward_kl = policy_sample_log_ratios.mean()

# (N=30000,): response IDs sampled from the reference policy
reference_sample_ids = torch.multinomial(
    reference_probs,
    num_samples,
    replacement=True,
    generator=torch.Generator().manual_seed(2403),
)

# (N=30000,): log-ratios evaluated on reference samples
reference_sample_log_ratios = log_ratio_by_response[reference_sample_ids]

# (): this estimates negative reverse KL, not forward KL
estimated_reference_average = reference_sample_log_ratios.mean()

torch.testing.assert_close(estimated_forward_kl, exact_forward_kl, atol=0.03, rtol=0)

torch.testing.assert_close(
    estimated_reference_average, -exact_reverse_kl, atol=0.03, rtol=0
)

print("Exact forward KL:", exact_forward_kl.item())
print("On-policy estimate:", estimated_forward_kl.item())
print("Reference-sampled mean:", estimated_reference_average.item())
print("Sampling-distribution checks passed.")

Exact forward KL: 0.40767741271438884
On-policy estimate: 0.4000379586224877
Reference-sampled mean: -0.29589537471084554
Sampling-distribution checks passed.


In [7]:
import plotly.graph_objects as go

sample_counts = [
    1,
    2,
    3,
    5,
    10,
    20,
    50,
    100,
    200,
    500,
    1000,
    2000,
    5000,
    10000,
    20000,
    num_samples,
]

# (K,): estimates using increasing numbers of policy samples
policy_running_means = [
    policy_sample_log_ratios[:count].mean().item() for count in sample_counts
]

# (K,): estimates using increasing numbers of reference samples
reference_running_means = [
    reference_sample_log_ratios[:count].mean().item() for count in sample_counts
]

kl_fig = go.Figure()

kl_fig.add_trace(
    go.Scatter(
        x=sample_counts,
        y=policy_running_means,
        mode="lines+markers",
        name="Samples from current policy",
    )
)

kl_fig.add_trace(
    go.Scatter(
        x=sample_counts,
        y=reference_running_means,
        mode="lines+markers",
        name="Samples from reference policy",
    )
)

kl_fig.add_hline(
    y=exact_forward_kl.item(),
    line_dash="dash",
    annotation_text="Exact forward KL",
)

kl_fig.add_hline(
    y=-exact_reverse_kl.item(),
    line_dash="dot",
    annotation_text="Negative reverse KL",
)

kl_fig.update_layout(
    title="Monte Carlo Log-Ratio Estimation",
    xaxis_title="Number of samples",
    yaxis_title="Mean sampled log-ratio",
    xaxis_type="log",
    height=470,
    hovermode="x unified",
)

kl_fig.show()

In [8]:
# (T=2,): current-policy token probabilities along one response
policy_token_probs = torch.tensor([0.8, 0.6], dtype=torch.float64)

# (T=2,): reference token probabilities along the same response
reference_token_probs = torch.tensor([0.5, 0.7], dtype=torch.float64)

# (T=2,): pointwise token log-probability ratios
token_log_ratios = policy_token_probs.log() - reference_token_probs.log()

# (): sum of per-token log-ratios
sequence_log_ratio = token_log_ratios.sum()

# (): direct ratio of complete-response probabilities
direct_sequence_log_ratio = (
    policy_token_probs.prod() / reference_token_probs.prod()
).log()

torch.testing.assert_close(sequence_log_ratio, direct_sequence_log_ratio)

# (Y=2,): KL-shaped rewards for the two-response toy environment
kl_shaping_rewards = -beta * log_ratio_by_response

# (Y=2,): task reward plus signed KL shaping
regularized_response_rewards = response_rewards + kl_shaping_rewards

# (): expected shaped return
expected_shaped_return = (policy_probs * regularized_response_rewards).sum()

# (): exact KL-regularized objective
exact_regularized_objective = (
    policy_probs * response_rewards
).sum() - beta * exact_forward_kl

torch.testing.assert_close(expected_shaped_return, exact_regularized_objective)

print("Token log-ratios:", token_log_ratios.tolist())
print("Sequence log-ratio:", sequence_log_ratio.item())
print("KL-shaped rewards:", kl_shaping_rewards.tolist())
print("Total shaped rewards:", regularized_response_rewards.tolist())
print("Expected shaped return:", expected_shaped_return.item())

Token log-ratios: [0.4700036292457356, -0.15415067982725827]
Sequence log-ratio: 0.3158529494184773
KL-shaped rewards: [0.2470143540220894, -0.7529856459779105]
Total shaped rewards: [0.2470143540220894, 0.2470143540220895]
Expected shaped return: 0.24701435402208943


In [9]:
# (Y=2,): snapshot of the rollout-time behavior policy
old_policy_probs = policy_probs.clone()

# (Y=2,): a hypothetical updated policy
new_policy_probs = torch.tensor([0.95, 0.05], dtype=torch.float64)

# (Y=2,): new-policy log-ratios against the fixed reference
new_log_ratios = new_policy_probs.log() - reference_probs.log()

# (): average new-policy log-ratio under OLD-policy samples
old_weighted_new_ratio = (old_policy_probs * new_log_ratios).sum()

# (): true forward KL of the NEW policy to the reference
exact_new_policy_kl = (new_policy_probs * new_log_ratios).sum()

# (Y=2,): importance weights converting old to new distribution
importance_weights = new_policy_probs / old_policy_probs

# (): importance-corrected expectation
importance_corrected_kl = (old_policy_probs * importance_weights * new_log_ratios).sum()

assert old_weighted_new_ratio < 0
assert exact_new_policy_kl > 0

torch.testing.assert_close(importance_corrected_kl, exact_new_policy_kl)

print("Old-sampled new log-ratio mean:", old_weighted_new_ratio.item())
print("True current-policy KL:", exact_new_policy_kl.item())
print("Importance-corrected KL:", importance_corrected_kl.item())

Old-sampled new log-ratio mean: -0.28281667855435544
True current-policy KL: 0.016706501178764648
Importance-corrected KL: 0.016706501178764648



### Reference Policy and Sampled KL — Takeaways

KL divergence is an expectation of pointwise log-probability
ratios, not the log-ratio of a single sampled response.

Individual sampled log-ratios can be negative, even though
the exact forward KL divergence is nonnegative.

Averaging log p(y) - log q(y) under samples from p estimates
D_KL(p || q). Averaging under samples from q instead estimates
-D_KL(q || p).

Monte Carlo estimation approaches the corresponding expectation
as the number of independent samples increases.

Autoregressive sequence log-ratios decompose into sums of
conditional token-level log-ratios along the same response.

A common RLHF construction uses signed KL-shaped token rewards:

$$
r_t^{\mathrm{KL}}
=
-\beta(\log\pi_{\mathrm{old}}(a_t\mid s_t)
-\log\pi_{\mathrm{ref}}(a_t\mid s_t))
$$

The expectation of the summed on-policy log-ratios equals the
sequence-level forward KL.

The PPO old policy is the behavior policy frozen for the
current update. The reference policy is the separately frozen
regularization baseline.

Evaluating current-policy log-ratios under old-policy samples
does not generally estimate current-policy KL without accounting
for the changed sampling distribution.

The next section will collect autoregressive GPT responses
and score them using a frozen reference policy and reward model.



## 03. Autoregressive GPT Rollouts and Reward Scoring

### What is a rollout?

In SFT, the target completion is supplied by the dataset.

In RLHF, the actor samples its own completion. We record the
actions and their probabilities before any PPO optimization.

```mermaid
flowchart TD
    A["Prompt x"] --> B["Actor: sample actions"]
    B --> C["Generated token IDs"]
    B --> D["Old action log-probabilities"]
    C --> E["Reference policy: score same actions"]
    C --> F["Frozen reward encoder + scalar head"]
    E --> G["Reference action log-probabilities"]
    F --> H["Sequence reward"]
    D --> I["Frozen rollout record"]
    G --> I
    H --> I
    C --> I
    I --> J["Next section: KL shaping and GAE"]
```

We use the complete, unfiltered next-token distribution:
temperature = 1, no top-k and no top-p filtering.

This matters because rollout-time action log-probabilities
must describe the actual behavior distribution.

The current actor and reference begin with identical SFT
weights. Therefore their initial log-probability ratios
should be approximately zero.

We stop sampling when a newline character appears or when
the generation budget is exhausted. Newline is an ordinary
character, not a native EOS token. We record whether a
response ended by newline or was truncated by the budget.


In [10]:
import copy
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from llmfp.data import CharacterTokenizer
from llmfp.generation import sample_next_token
from llmfp.models import GPT, GPTConfig
from llmfp.rl import rescore_actions_from_logits
from llmfp.training import bradley_terry_loss, completion_sequence_log_probs

checkpoint_path = Path("../checkpoints/tiny_shakespeare_sft/step_00040.pt")

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

tokenizer = CharacterTokenizer(checkpoint["tokenizer_vocabulary"])

config = GPTConfig(**checkpoint["model_config"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Restore the SFT model as a frozen reference policy.
reference_policy = GPT(config).to(device)

reference_policy.load_state_dict(checkpoint["model_state_dict"], strict=True)

reference_policy.requires_grad_(False)
reference_policy.eval()

# Create an independent trainable actor with identical weights.
actor = copy.deepcopy(reference_policy)
actor.requires_grad_(True)
actor.eval()

assert tokenizer.vocab_size == config.vocab_size == 65

assert not any(parameter.requires_grad for parameter in reference_policy.parameters())

assert all(parameter.requires_grad for parameter in actor.parameters())

assert (
    actor.token_embedding.weight.data_ptr()
    != reference_policy.token_embedding.weight.data_ptr()
)

# (B=1, T_prompt): a shared input for checking initial agreement
probe_ids = torch.tensor(
    [tokenizer.encode("User: Say yes\nAssistant: ")], dtype=torch.long, device=device
)

with torch.no_grad():
    # (B=1, T_prompt, V): two independently computed logits tensors
    actor_probe_logits = actor(probe_ids)
    reference_probe_logits = reference_policy(probe_ids)

torch.testing.assert_close(actor_probe_logits, reference_probe_logits)

print("Actor and Reference initialized identically.")
print("Actor trainable:", any(p.requires_grad for p in actor.parameters()))
print(
    "Reference trainable:", any(p.requires_grad for p in reference_policy.parameters())
)
print("Device:", device)


Actor and Reference initialized identically.
Actor trainable: True
Reference trainable: False
Device: cuda


In [ ]:
def pad_full_sequences(
    sequences: list[torch.Tensor],
    pad_token_id: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Right-pad complete token sequences without shifting targets.

    Args:
        sequences: Nonempty list of (L_i,) token ID tensors.
        pad_token_id: Valid character token used for right padding.

    Returns:
        padded: (B, L_max) token IDs.
        lengths: (B,) number of real tokens in each sequence.
    """
    sequence_device = sequences[0].device

    # (B,): number of real tokens in each complete sequence
    lengths = torch.tensor(
        [sequence.numel() for sequence in sequences],
        dtype=torch.long,
        device=sequence_device,
    )

    # (B, L_max): right-padded complete sequences
    padded = torch.full(
        (len(sequences), int(lengths.max().item())),
        pad_token_id,
        dtype=torch.long,
        device=sequence_device,
    )

    for row, sequence in enumerate(sequences):
        padded[row, : sequence.numel()] = sequence

    return padded, lengths


@torch.no_grad()
def frozen_last_token_features(
    backbone: GPT,
    full_token_ids: torch.Tensor,
    lengths: torch.Tensor,
) -> torch.Tensor:
    """Extract the last real-token features of a frozen causal GPT.

    Args:
        backbone: Frozen GPT model in evaluation mode.
        full_token_ids: (B, L_max) right-padded complete sequences.
        lengths: (B,) positive real sequence lengths.

    Returns:
        (B, C) contextual features at final real-token positions.
    """
    # (B, L_max, C): token embeddings
    hidden = backbone.token_embedding(full_token_ids)

    for block in backbone.blocks:
        # (B, L_max, C): updated causal hidden representations
        hidden = block(hidden)

    # (B, L_max, C): normalized final hidden representations
    hidden = backbone.final_norm(hidden)

    # (B,): indices selecting individual batch rows
    row_indices = torch.arange(full_token_ids.shape[0], device=full_token_ids.device)

    # (B, C): final real-token features, excluding padding
    return hidden[row_indices, lengths - 1]


In [12]:
reward_training_pairs = [
    ("Say yes", "Yes", "No"),
    ("Say no", "No", "Yes"),
    ("Say hello", "Hello", "Farewell"),
]

pad_token_id = tokenizer.token_to_id[" "]
newline_token_id = tokenizer.token_to_id["\n"]

reward_training_sequences = []

for instruction, chosen, rejected in reward_training_pairs:
    for response in (chosen, rejected):
        full_text = f"User: {instruction}\nAssistant: {response}\n"

        # (L_i,): full prompt-response token sequence
        token_ids = torch.tensor(
            tokenizer.encode(full_text), dtype=torch.long, device=device
        )

        reward_training_sequences.append(token_ids)

# (B=6, L_max): right-padded preference candidates
reward_train_inputs, reward_train_lengths = pad_full_sequences(
    reward_training_sequences, pad_token_id
)

# (B=6, C=256): fixed SFT representations of preference candidates
reward_train_features = frozen_last_token_features(
    reference_policy, reward_train_inputs, reward_train_lengths
)

reward_head = nn.Linear(config.embedding_dim, 1, bias=False).to(device)

nn.init.zeros_(reward_head.weight)

reward_optimizer = torch.optim.AdamW(
    reward_head.parameters(), lr=1e-2, weight_decay=0.0
)

# (B=6,): initially equal scalar scores
initial_scores = reward_head(reward_train_features).squeeze(-1)

# (): initial mean Bradley-Terry loss
initial_bt_loss = bradley_terry_loss(initial_scores[0::2], initial_scores[1::2])

for step in range(80):
    # (B=6,): trainable reward scores
    scores = reward_head(reward_train_features).squeeze(-1)

    # (): chosen-over-rejected pairwise objective
    loss = bradley_terry_loss(scores[0::2], scores[1::2])

    reward_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    reward_optimizer.step()

with torch.no_grad():
    # (B=6,): final scores after reward-head training
    final_scores = reward_head(reward_train_features).squeeze(-1)

    # (B_pairs=3,): learned preference margins
    trained_reward_margins = final_scores[0::2] - final_scores[1::2]

    # (): final training objective
    final_bt_loss = bradley_terry_loss(final_scores[0::2], final_scores[1::2])

assert final_bt_loss < initial_bt_loss
assert torch.isfinite(final_scores).all()

# Freeze the reward head before collecting RLHF data.
reward_head.requires_grad_(False)
reward_head.eval()

print("Initial BT loss:", initial_bt_loss.item())
print("Final BT loss:", final_bt_loss.item())
print("Training reward margins:", trained_reward_margins.tolist())
print("Reward head frozen:", not any(p.requires_grad for p in reward_head.parameters()))


Initial BT loss: 0.6931471824645996
Final BT loss: 0.01911216601729393
Training reward margins: [3.52864933013916, 3.6016359329223633, 6.501101493835449]
Reward head frozen: True


In [13]:
@torch.no_grad()
def collect_one_response(
    policy: GPT,
    instruction: str,
    *,
    max_new_tokens: int,
    generator: torch.Generator,
) -> dict:
    """Sample a response and record behavior-policy action log-probs.

    Args:
        policy: Actor held fixed during rollout collection.
        instruction: Text instruction for the character tokenizer.
        max_new_tokens: Maximum generated character count.
        generator: Sampling RNG for the actor's device.

    Returns:
        Full sampled token IDs, prompt length, action log-probs,
        and whether newline stopped generation.
    """
    policy_device = next(policy.parameters()).device

    prompt_text = f"User: {instruction}\nAssistant: "

    # (1, P): prompt token IDs
    prompt_ids = torch.tensor(
        [tokenizer.encode(prompt_text)], dtype=torch.long, device=policy_device
    )

    full_ids = prompt_ids.clone()

    # Prefill: logits for the next token after the prompt.
    next_logits, cache = policy(prompt_ids, use_cache=True)

    action_log_probs = []
    ended_by_newline = False

    for step in range(max_new_tokens):
        # (1, V): next-token logits under the behavior policy
        action_logits = next_logits[:, -1, :]

        # (1, 1): sample from the full unfiltered policy distribution
        sampled_id = sample_next_token(action_logits, generator=generator)

        # (): log-probability of the sampled action
        sampled_log_prob = (
            F
            .log_softmax(action_logits, dim=-1)
            .gather(dim=-1, index=sampled_id)
            .squeeze()
        )

        action_log_probs.append(sampled_log_prob)

        # (1, P+t+1): append the sampled action
        full_ids = torch.cat([full_ids, sampled_id], dim=1)

        if sampled_id.item() == newline_token_id:
            ended_by_newline = True
            break

        if step < max_new_tokens - 1:
            # (1, 1, V): incrementally decode the next position
            next_logits, cache = policy(
                sampled_id, past_key_values=cache, use_cache=True
            )

    # (T_actions,): frozen collection-time action log-probabilities
    collected_log_probs = torch.stack(action_log_probs).detach()

    return {
        "instruction": instruction,
        "token_ids": full_ids.squeeze(0).detach(),
        "prompt_length": prompt_ids.shape[1],
        "old_log_probs": collected_log_probs,
        "ended_by_newline": ended_by_newline,
    }


In [14]:
rollout_instructions = ["Say yes", "Say no", "Say hello", "Say welcome"]

# Two independent samples per instruction.
rollout_prompts = [
    instruction for instruction in rollout_instructions for _ in range(2)
]

rollout_generator = torch.Generator(device=device).manual_seed(2403)

rollouts = [
    collect_one_response(
        actor, instruction, max_new_tokens=16, generator=rollout_generator
    )
    for instruction in rollout_prompts
]

for index, rollout in enumerate(rollouts):
    # (T_actions,): generated action token IDs
    generated_ids = rollout["token_ids"][rollout["prompt_length"] :]

    response_text = tokenizer.decode(generated_ids.cpu().tolist())

    print(
        f"{index:02d} | "
        f"{rollout['instruction']:12s} | "
        f"{response_text!r:20s} | "
        f"actions={generated_ids.numel():2d} | "
        f"newline={rollout['ended_by_newline']}"
    )

assert len(rollouts) == 8
assert all(rollout["old_log_probs"].numel() > 0 for rollout in rollouts)


00 | Say yes      | 'Yes\n'              | actions= 4 | newline=True
01 | Say yes      | 'Yes\n'              | actions= 4 | newline=True
02 | Say no       | 'Yes\n'              | actions= 4 | newline=True
03 | Say no       | 'Yes\n'              | actions= 4 | newline=True
04 | Say hello    | 'Hello\n'            | actions= 6 | newline=True
05 | Say hello    | 'Hello\n'            | actions= 6 | newline=True
06 | Say welcome  | 'Thanks\n'           | actions= 7 | newline=True
07 | Say welcome  | 'Thanks\n'           | actions= 7 | newline=True


In [15]:
# (B, L_max): padded prompt + generated response IDs
rollout_token_ids, rollout_lengths = pad_full_sequences(
    [rollout["token_ids"] for rollout in rollouts], pad_token_id
)

rollout_batch_size = len(rollouts)

max_actions = max(rollout["old_log_probs"].numel() for rollout in rollouts)

# (B, T_max): valid generated-action positions
rollout_action_mask = torch.zeros(
    (rollout_batch_size, max_actions), dtype=torch.bool, device=device
)

# (B, T_max): absolute token indices of generated actions
rollout_action_positions = torch.zeros(
    (rollout_batch_size, max_actions), dtype=torch.long, device=device
)

# (B, T_max): collection-time sampled action log-probabilities
old_action_log_probs = torch.zeros(
    (rollout_batch_size, max_actions),
    dtype=next(actor.parameters()).dtype,
    device=device,
)

for row, rollout in enumerate(rollouts):
    start = rollout["prompt_length"]
    count = rollout["old_log_probs"].numel()

    assert rollout["token_ids"].numel() == start + count

    rollout_action_mask[row, :count] = True

    rollout_action_positions[row, :count] = torch.arange(
        start, start + count, device=device
    )

    old_action_log_probs[row, :count] = rollout["old_log_probs"]

# (B,): whether newline ended each sampled response
rollout_newline_ended = torch.tensor(
    [rollout["ended_by_newline"] for rollout in rollouts],
    dtype=torch.bool,
    device=device,
)

assert rollout_action_mask.sum().item() == sum(
    rollout["old_log_probs"].numel() for rollout in rollouts
)

assert torch.all(old_action_log_probs[~rollout_action_mask] == 0)

print("Full token batch:", rollout_token_ids.shape)
print("Action positions:", rollout_action_positions.shape)
print("Action mask:", rollout_action_mask.shape)
print("Old log-probs:", old_action_log_probs.shape)
print("Newline-terminated:", rollout_newline_ended.sum().item())


Full token batch: torch.Size([8, 36])
Action positions: torch.Size([8, 7])
Action mask: torch.Size([8, 7])
Old log-probs: torch.Size([8, 7])
Newline-terminated: 8


In [16]:
with torch.no_grad():
    # (B, L_max, V): actor logits for full collected sequences
    actor_rollout_logits = actor(rollout_token_ids)

    # (B, L_max, V): frozen reference logits for same sequences
    reference_rollout_logits = reference_policy(rollout_token_ids)

    # (B, T_max): actor log-probs of stored sampled actions
    rescored_actor_log_probs = rescore_actions_from_logits(
        actor_rollout_logits,
        rollout_token_ids,
        rollout_action_positions,
        rollout_action_mask,
    )

    # (B, T_max): reference log-probs of the SAME sampled actions
    reference_action_log_probs = rescore_actions_from_logits(
        reference_rollout_logits,
        rollout_token_ids,
        rollout_action_positions,
        rollout_action_mask,
    )

torch.testing.assert_close(
    rescored_actor_log_probs, old_action_log_probs, rtol=1e-4, atol=1e-4
)

# At initialization, the actor and reference have identical weights.
torch.testing.assert_close(
    rescored_actor_log_probs, reference_action_log_probs, rtol=1e-4, atol=1e-4
)

# (B, L_max): pre-aligned labels for full-response scoring
rollout_labels = torch.full_like(rollout_token_ids, -100)

for row in range(rollout_batch_size):
    # (T_i,): real action positions for this generated response
    action_positions = rollout_action_positions[row, rollout_action_mask[row]]

    # Predictor at position j-1 assigns probability to token j.
    rollout_labels[
        row,
        action_positions - 1,
    ] = rollout_token_ids[row, action_positions]

with torch.no_grad():
    # (B,): complete-response log-probabilities from aligned labels
    sequence_log_probs = completion_sequence_log_probs(
        actor_rollout_logits, rollout_labels
    )

    # (B,): same sequence scores from collection-time log-probs
    summed_action_log_probs = old_action_log_probs.sum(dim=1)

torch.testing.assert_close(
    sequence_log_probs, summed_action_log_probs, rtol=1e-4, atol=1e-4
)

print("Sampling vs teacher-forced action log-probs: PASSED")
print("Action sums vs sequence log-probs: PASSED")
print("Initial actor/reference agreement: PASSED")


Sampling vs teacher-forced action log-probs: PASSED
Action sums vs sequence log-probs: PASSED
Initial actor/reference agreement: PASSED


In [17]:
with torch.no_grad():
    # (B, C=256): frozen final-token contextual representations
    rollout_features = frozen_last_token_features(
        reference_policy, rollout_token_ids, rollout_lengths
    )

    # (B,): learned reward scores of sampled sequences
    rollout_reward_scores = reward_head(rollout_features).squeeze(-1)

    # (B, T_max): signed collection-time token log-ratios
    rollout_token_log_ratios = old_action_log_probs - reference_action_log_probs

    # (B,): signed sequence log-ratios
    rollout_sequence_log_ratios = rollout_token_log_ratios.sum(dim=1)

assert rollout_reward_scores.shape == (rollout_batch_size,)
assert torch.isfinite(rollout_reward_scores).all()

assert torch.all(rollout_token_log_ratios[~rollout_action_mask] == 0)

# Both policies are still identical: initial reference KL contribution is zero.
torch.testing.assert_close(
    rollout_sequence_log_ratios,
    torch.zeros_like(rollout_sequence_log_ratios),
    rtol=0,
    atol=1e-4,
)

for index, rollout in enumerate(rollouts):
    print(
        f"{rollout['instruction']:12s} | "
        f"reward={rollout_reward_scores[index].item():+8.4f} | "
        f"logP={sequence_log_probs[index].item():+9.4f} | "
        f"log-ratio={rollout_sequence_log_ratios[index].item():+.5f}"
    )

print("Frozen reward scoring passed.")


Say yes      | reward= +6.5645 | logP=  -0.0007 | log-ratio=+0.00000
Say yes      | reward= +6.5645 | logP=  -0.0007 | log-ratio=+0.00000
Say no       | reward= +4.3336 | logP=  -0.0015 | log-ratio=+0.00000
Say no       | reward= +4.3336 | logP=  -0.0015 | log-ratio=+0.00000
Say hello    | reward= +3.0029 | logP=  -0.0011 | log-ratio=+0.00000
Say hello    | reward= +3.0029 | logP=  -0.0011 | log-ratio=+0.00000
Say welcome  | reward= +1.4997 | logP=  -0.2762 | log-ratio=-0.00000
Say welcome  | reward= +1.4997 | logP=  -0.2762 | log-ratio=-0.00000
Frozen reward scoring passed.


In [18]:
import plotly.graph_objects as go

sample_index = 0

# (T_i,): real generated-token IDs for the selected response
sample_action_ids = rollouts[sample_index]["token_ids"][
    rollouts[sample_index]["prompt_length"] :
]

sample_characters = list(tokenizer.decode(sample_action_ids.cpu().tolist()))

# (T_i,): collection-time action log-probabilities
sample_old_log_probs = (
    old_action_log_probs[
        sample_index,
        rollout_action_mask[sample_index],
    ]
    .cpu()
    .tolist()
)

# (T_i,): independently rescored reference log-probabilities
sample_reference_log_probs = (
    reference_action_log_probs[
        sample_index,
        rollout_action_mask[sample_index],
    ]
    .cpu()
    .tolist()
)

token_fig = go.Figure()

token_fig.add_trace(
    go.Bar(
        x=list(range(1, len(sample_characters) + 1)),
        y=sample_old_log_probs,
        name="Collected actor logP",
        customdata=[repr(char) for char in sample_characters],
        hovertemplate=(
            "Action position: %{x}"
            "<br>Character: %{customdata}"
            "<br>Actor logP: %{y:.5f}"
            "<extra></extra>"
        ),
    )
)

token_fig.add_trace(
    go.Scatter(
        x=list(range(1, len(sample_characters) + 1)),
        y=sample_reference_log_probs,
        mode="lines+markers",
        name="Reference logP",
    )
)

token_fig.update_layout(
    title=f"Sampled Response — {rollouts[sample_index]['instruction']}",
    xaxis_title="Generated character position",
    yaxis_title="Conditional token log-probability",
    xaxis=dict(dtick=1),
    height=430,
)

token_fig.show()



### Autoregressive GPT Rollouts — Takeaways

We restored the SFT checkpoint into two independent GPT models:
a trainable actor and a frozen reference policy.

Because Lesson 23 did not save a reward-model checkpoint,
we reconstructed a frozen GPT feature encoder and retrained
a scalar reward head on the original three synthetic pairs.

We collected autoregressive responses using the actor's full,
unfiltered next-token sampling distribution.

Each rollout records its complete token sequence, prompt
length, generated action positions, action mask, and
collection-time action log-probabilities.

We re-scored the same sampled actions by teacher forcing and
verified agreement with the log-probabilities saved at sampling.

We also verified that summing action log-probabilities equals
completion sequence log-probability.

The reference policy scored the same generated actions.

At initialization, actor and reference weights are identical,
so their sampled token log-ratios are approximately zero.

The frozen reward head assigned one scalar score to each
sampled response or truncated prefix.

A newline character ends generation when observed; exhausting
the token budget instead produces a truncated rollout.
These cases must remain distinguishable.

The next section will distribute KL penalties across actions,
attach sequence-level reward to an appropriate boundary,
and compute Critic value targets and GAE.



## 04. Token-Level KL Rewards and GAE

### From a sequence reward to action-level learning signals

The reward model scores a completed response, but PPO updates
the probabilities of individual generated actions.

For each sampled action:

$$
r_t^{KL}
=
-\beta[
\log\pi_{\mathrm{old}}(a_t\mid s_t)
-
\log\pi_{\mathrm{ref}}(a_t\mid s_t)
]
$$

For a response that terminates normally:

$$
r_t =
r_t^{KL}
+
\mathbf{1}[t=T-1]R_\phi(x,y)
$$

The reward model score is added only to the final valid
action. Earlier actions can still receive credit through
future returns and advantage estimation.

```mermaid
flowchart TD
    A["Frozen rollout: sampled tokens"] --> B["Old and reference action logP"]
    B --> C["Token-level KL-shaped rewards"]
    A --> D["Frozen reward model"]
    D --> E["Terminal sequence reward"]
    C --> F["Per-action RLHF rewards"]
    E --> F
    A --> G["Trainable Critic: V(s)"]
    F --> H["GAE"]
    G --> H
    H --> I["Advantages and value targets"]
    I --> J["Next: PPO Actor-Critic update"]
```

We distinguish normal newline termination from a generation
budget cutoff.

In this toy environment, newline is the terminal marker.
A budget cutoff is treated as truncation of an episode that
would otherwise continue: no terminal reward is attached,
and the Critic may bootstrap from the final observed state.

This convention is a modeling choice, not a universal rule
for all LLM generation tasks.


In [19]:
import torch

kl_beta = 0.1

# (T=3,): illustrative sampled action log-ratios
toy_log_ratios = torch.tensor([0.2, -0.1, 0.4])

# (): illustrative terminal reward-model score
toy_terminal_reward = torch.tensor(2.0)

# (T=3,): signed KL-shaped token rewards
toy_kl_rewards = -kl_beta * toy_log_ratios

# (T=3,): token rewards before adding the terminal score
toy_shaped_rewards = toy_kl_rewards.clone()

# Add reward-model score only at the final action.
toy_shaped_rewards[-1] += toy_terminal_reward

# (): sum of action rewards
toy_return = toy_shaped_rewards.sum()

# (): equivalent sequence-level regularized reward
toy_expected_return = toy_terminal_reward - kl_beta * toy_log_ratios.sum()

torch.testing.assert_close(toy_return, toy_expected_return)

print("KL rewards:", toy_kl_rewards.tolist())
print("Shaped rewards:", toy_shaped_rewards.tolist())
print("Total return:", toy_return.item())


KL rewards: [-0.020000001415610313, 0.010000000707805157, -0.04000000283122063]
Shaped rewards: [-0.020000001415610313, 0.010000000707805157, 1.9600000381469727]
Total return: 1.9500000476837158


In [20]:
# (B=8, T_max=7): rollout-time old-minus-reference log-ratios
collected_log_ratios = old_action_log_probs - reference_action_log_probs

# (B=8, T_max=7): signed KL rewards for actual generated actions
kl_action_rewards = (-kl_beta * collected_log_ratios).masked_fill(
    ~rollout_action_mask, 0.0
)

# (B=8, T_max=7): initially zero terminal-reward component
terminal_action_rewards = torch.zeros_like(kl_action_rewards)

# (B=8,): number of valid sampled actions per rollout
action_counts = rollout_action_mask.sum(dim=1)

for row in range(rollout_batch_size):
    last_action_index = int(action_counts[row].item()) - 1

    if rollout_newline_ended[row].item():
        terminal_action_rewards[
            row,
            last_action_index,
        ] = rollout_reward_scores[row]

# (B=8, T_max=7): complete action-level RLHF rewards
rlhf_action_rewards = kl_action_rewards + terminal_action_rewards

# (B=8,): sum of shaped rewards for each trajectory
episode_returns = rlhf_action_rewards.sum(dim=1)

# (B=8,): reward scores used only for normally terminated responses
eligible_terminal_rewards = rollout_reward_scores * rollout_newline_ended.to(
    rollout_reward_scores.dtype
)

# (B=8,): equivalent sequence-level shaped returns
expected_episode_returns = (
    eligible_terminal_rewards - kl_beta * collected_log_ratios.sum(dim=1)
)

torch.testing.assert_close(
    episode_returns, expected_episode_returns, atol=1e-5, rtol=1e-5
)

assert torch.all(rlhf_action_rewards[~rollout_action_mask] == 0)

for row, rollout in enumerate(rollouts):
    print(
        f"{rollout['instruction']:12s} | "
        f"actions={action_counts[row].item():2d} | "
        f"terminated={rollout_newline_ended[row].item()} | "
        f"return={episode_returns[row].item():+.4f}"
    )

print("Action-level reward shaping passed.")


Say yes      | actions= 4 | terminated=True | return=+6.5645
Say yes      | actions= 4 | terminated=True | return=+6.5645
Say no       | actions= 4 | terminated=True | return=+4.3336
Say no       | actions= 4 | terminated=True | return=+4.3336
Say hello    | actions= 6 | terminated=True | return=+3.0029
Say hello    | actions= 6 | terminated=True | return=+3.0029
Say welcome  | actions= 7 | terminated=True | return=+1.4997
Say welcome  | actions= 7 | terminated=True | return=+1.4997
Action-level reward shaping passed.


In [21]:
import copy

import torch.nn as nn

from llmfp.rl import (
    RolloutBatch,
    align_state_values_for_actions,
    generalized_advantage_estimation,
)


class RLHFValueCritic(nn.Module):
    """Estimate state values with an independent GPT backbone.

    Args:
        policy: GPT supplying the initial representation weights.
    """

    def __init__(self, policy: GPT) -> None:
        super().__init__()

        self.token_embedding = copy.deepcopy(policy.token_embedding)
        self.blocks = copy.deepcopy(policy.blocks)
        self.final_norm = copy.deepcopy(policy.final_norm)

        self.value_head = nn.Linear(policy.config.embedding_dim, 1)

        # Initial state-value predictions are zero.
        nn.init.zeros_(self.value_head.weight)
        nn.init.zeros_(self.value_head.bias)

        # The source reference was frozen; the critic must be trainable.
        self.requires_grad_(True)

    def forward(
        self,
        token_ids: torch.Tensor,
    ) -> torch.Tensor:
        """Return one value prediction per token position.

        Args:
            token_ids: (B, L) complete or right-padded sequences.

        Returns:
            (B, L) causal state-value estimates.
        """
        # (B, L, C): embedded token sequence
        hidden = self.token_embedding(token_ids)

        for block in self.blocks:
            # (B, L, C): causal contextual representations
            hidden = block(hidden)

        # (B, L, C): normalized Transformer output
        hidden = self.final_norm(hidden)

        # (B, L): one predicted state value per token position
        return self.value_head(hidden).squeeze(-1)


critic = RLHFValueCritic(reference_policy).to(device)
critic.eval()

assert all(parameter.requires_grad for parameter in critic.parameters())

assert (
    critic.token_embedding.weight.data_ptr()
    != reference_policy.token_embedding.weight.data_ptr()
)

print("Critic initialized independently.")
print("Critic trainable:", any(p.requires_grad for p in critic.parameters()))


Critic initialized independently.
Critic trainable: True


In [22]:
with torch.no_grad():
    # (B=8, L_max=36): one Critic value per supplied token position
    values_by_token = critic(rollout_token_ids)

    # (B=8, T_max+1=8): aligned V(s_0), ..., V(s_T)
    rollout_state_values = align_state_values_for_actions(
        values_by_token, rollout_action_positions, rollout_action_mask
    )

# (B=8, T_max=7): whether the next-state value can be used
bootstrap_mask = torch.zeros_like(rlhf_action_rewards)

# (B=8, T_max=7): whether GAE recurses to the next collected action
continuation_mask = torch.zeros_like(rlhf_action_rewards)

for row in range(rollout_batch_size):
    count = int(action_counts[row].item())

    # All actions except the last observed action have successors.
    if count > 1:
        bootstrap_mask[row, : count - 1] = 1.0
        continuation_mask[row, : count - 1] = 1.0

    # A truncated episode may bootstrap from the final observed state.
    # A naturally terminated episode may not.
    if not rollout_newline_ended[row].item():
        bootstrap_mask[row, count - 1] = 1.0

    # The collected trajectory stops here in both cases.
    continuation_mask[row, count - 1] = 0.0

assert rollout_state_values.shape == (rollout_batch_size, max_actions + 1)

# The value head starts at zero everywhere.
torch.testing.assert_close(
    rollout_state_values,
    torch.zeros_like(rollout_state_values),
)

assert torch.all(continuation_mask <= bootstrap_mask)

assert torch.all(bootstrap_mask[~rollout_action_mask] == 0)

assert torch.all(continuation_mask[~rollout_action_mask] == 0)

print("Aligned Critic values:", rollout_state_values.shape)
print("Bootstrap mask:", bootstrap_mask.shape)
print("Continuation mask:", continuation_mask.shape)
print("Critic alignment and boundary checks passed.")


Aligned Critic values: torch.Size([8, 8])
Bootstrap mask: torch.Size([8, 7])
Continuation mask: torch.Size([8, 7])
Critic alignment and boundary checks passed.


In [23]:
gamma = 1.0
gae_lambda = 1.0

# (B=8, T_max=7): estimated action advantages
rollout_advantages = torch.zeros_like(rlhf_action_rewards)

# (B=8, T_max=7): targets for the trainable Critic
rollout_value_targets = torch.zeros_like(rlhf_action_rewards)

for row in range(rollout_batch_size):
    # (T_max,): one sampled trajectory's advantages
    advantages, targets = generalized_advantage_estimation(
        rewards=rlhf_action_rewards[row],
        values=rollout_state_values[row],
        bootstrap_mask=bootstrap_mask[row],
        continuation_mask=continuation_mask[row],
        gamma=gamma,
        gae_lambda=gae_lambda,
    )

    rollout_advantages[row] = advantages
    rollout_value_targets[row] = targets

# Since the initial Critic predicts zero and lambda=gamma=1,
# each advantage equals the undiscounted future shaped return.
for row in range(rollout_batch_size):
    count = int(action_counts[row].item())

    # (T_i,): direct Monte Carlo returns for valid actions
    valid_rewards = rlhf_action_rewards[row, :count]

    direct_returns = torch.flip(
        torch.cumsum(
            torch.flip(valid_rewards, dims=[0]),
            dim=0,
        ),
        dims=[0],
    )

    torch.testing.assert_close(
        rollout_advantages[row, :count], direct_returns, atol=1e-5, rtol=1e-5
    )

torch.testing.assert_close(
    rollout_value_targets,
    rollout_advantages + rollout_state_values[:, :-1],
)

assert torch.all(rollout_advantages[~rollout_action_mask] == 0)

assert torch.isfinite(rollout_advantages).all()

print("GAE shape:", rollout_advantages.shape)
print("Value targets shape:", rollout_value_targets.shape)
print("GAE and Monte Carlo return equivalence passed.")


GAE shape: torch.Size([8, 7])
Value targets shape: torch.Size([8, 7])
GAE and Monte Carlo return equivalence passed.


In [24]:
# (T=2,): illustrative immediate rewards
boundary_rewards = torch.tensor([0.0, 0.0])

# (T+1=3,): illustrative state values including the final observed state
boundary_values = torch.tensor([0.2, 0.4, 0.9])

# (T=2,): natural termination disables final bootstrapping
terminal_bootstrap = torch.tensor([1.0, 0.0])

# (T=2,): time-limit truncation preserves final bootstrapping
truncated_bootstrap = torch.tensor([1.0, 1.0])

# (T=2,): both collected trajectories stop after their second action
boundary_continuation = torch.tensor([1.0, 0.0])

terminal_advantages, terminal_targets = generalized_advantage_estimation(
    rewards=boundary_rewards,
    values=boundary_values,
    bootstrap_mask=terminal_bootstrap,
    continuation_mask=boundary_continuation,
    gamma=1.0,
    gae_lambda=1.0,
)

truncated_advantages, truncated_targets = generalized_advantage_estimation(
    rewards=boundary_rewards,
    values=boundary_values,
    bootstrap_mask=truncated_bootstrap,
    continuation_mask=boundary_continuation,
    gamma=1.0,
    gae_lambda=1.0,
)

torch.testing.assert_close(terminal_targets, torch.zeros_like(terminal_targets))

torch.testing.assert_close(truncated_targets, torch.full_like(truncated_targets, 0.9))

print("Terminal value targets:", terminal_targets.tolist())
print("Truncated value targets:", truncated_targets.tolist())
print("Termination and truncation semantics passed.")


Terminal value targets: [0.0, 0.0]
Truncated value targets: [0.8999999761581421, 0.8999999761581421]
Termination and truncation semantics passed.


In [25]:
rlhf_rollout = RolloutBatch(
    token_ids=rollout_token_ids.detach().clone(),
    action_token_positions=rollout_action_positions.detach().clone(),
    action_mask=rollout_action_mask.detach().clone(),
    old_log_probabilities=old_action_log_probs.detach().clone(),
    rewards=rlhf_action_rewards.detach().clone(),
    rollout_values=rollout_state_values.detach().clone(),
    bootstrap_mask=bootstrap_mask.detach().clone(),
    continuation_mask=continuation_mask.detach().clone(),
    advantages=rollout_advantages.detach().clone(),
    value_targets=rollout_value_targets.detach().clone(),
)

assert rlhf_rollout.token_ids.shape == rollout_token_ids.shape

assert rlhf_rollout.advantages.shape == (rollout_batch_size, max_actions)

assert not rlhf_rollout.advantages.requires_grad
assert not rlhf_rollout.old_log_probabilities.requires_grad
assert not rlhf_rollout.value_targets.requires_grad

print("Frozen RLHF rollout successfully constructed.")
print("Token IDs:", rlhf_rollout.token_ids.shape)
print("Action rewards:", rlhf_rollout.rewards.shape)
print("Advantages:", rlhf_rollout.advantages.shape)
print("Value targets:", rlhf_rollout.value_targets.shape)


Frozen RLHF rollout successfully constructed.
Token IDs: torch.Size([8, 36])
Action rewards: torch.Size([8, 7])
Advantages: torch.Size([8, 7])
Value targets: torch.Size([8, 7])


In [26]:
import plotly.graph_objects as go

sample_row = 2
sample_count = int(action_counts[sample_row].item())

# (T_max,): GAE with lambda=0.95 for comparison
lambda95_advantages, _ = generalized_advantage_estimation(
    rewards=rlhf_action_rewards[sample_row],
    values=rollout_state_values[sample_row],
    bootstrap_mask=bootstrap_mask[sample_row],
    continuation_mask=continuation_mask[sample_row],
    gamma=1.0,
    gae_lambda=0.95,
)

# (T_i,): generated character IDs for this sampled response
sample_token_ids = rollout_token_ids[
    sample_row,
    rollout_action_positions[sample_row, :sample_count],
]

sample_characters = tokenizer.decode(sample_token_ids.cpu().tolist())

positions = list(range(1, sample_count + 1))

credit_fig = go.Figure()

credit_fig.add_trace(
    go.Bar(
        x=positions,
        y=rlhf_action_rewards[sample_row, :sample_count].cpu().tolist(),
        name="Immediate shaped reward",
    )
)

credit_fig.add_trace(
    go.Scatter(
        x=positions,
        y=rollout_advantages[sample_row, :sample_count].cpu().tolist(),
        name="GAE advantage (lambda=1)",
        mode="lines+markers",
    )
)

credit_fig.add_trace(
    go.Scatter(
        x=positions,
        y=lambda95_advantages[:sample_count].cpu().tolist(),
        name="GAE advantage (lambda=0.95)",
        mode="lines+markers",
        line=dict(dash="dash"),
    )
)

credit_fig.update_layout(
    title=f"Temporal Credit Assignment — {rollouts[sample_row]['instruction']}: {sample_characters!r}",
    xaxis_title="Generated character position",
    yaxis_title="Reward / estimated advantage",
    xaxis=dict(
        tickmode="array",
        tickvals=positions,
        ticktext=[f"{i}: {char!r}" for i, char in zip(positions, sample_characters)],
    ),
    height=460,
    hovermode="x unified",
)

credit_fig.show()



### Token-Level KL Rewards and GAE — Takeaways

We converted rollout-time old-minus-reference log-probability
ratios into signed KL-shaped action rewards.

A frozen reward model supplied one sequence-level scalar.
For normally terminated responses, we attached this score to
the final valid generated action.

The sum of shaped action rewards equals the corresponding
sequence-level reward minus the sampled KL penalty.

We constructed an independent, trainable Transformer Critic
initialized from SFT representations with a zero value head.

The Critic estimates future RLHF returns from partial
generation states. It is not the preference reward model.

We aligned state values with sampled actions and their
successor states.

Separate bootstrap and continuation masks distinguish
natural termination from a truncation boundary.

We reused the established GAE function to compute
per-action advantages and value targets.

With gamma=1, lambda=1, and initial zero value estimates,
GAE agrees with undiscounted Monte Carlo returns.

The GAE lambda parameter changes temporal credit assignment
when value estimates are imperfect.

We packaged immutable rollout-time facts and fixed targets
in the existing RolloutBatch structure.

The initial Critic is untrained. A positive estimated
advantage relative to its zero prediction is not evidence
that a sampled response is better than the policy average.

The next section uses these fixed rollout targets for
PPO Actor-Critic optimization.
